# Day 08 â€” Structured Outputs, Reliability & API Deployment (hands-on)

Companion notebook to [`../notes.md`](../notes.md). Sections 1-3 run fully **without any API key** â€”
they're about validation and retry mechanics, not model quality. Section 4 is optional and needs a key.
Section 5 (the actual API) lives in [`api_example.py`](api_example.py) since FastAPI runs as a server,
not notebook cells.

## 1. Schema-first structured output with Pydantic

Define the exact shape you want *before* asking anything â€” field names, types, and allowed values.

In [1]:
from enum import Enum
from pydantic import BaseModel, Field, ValidationError


class IssueType(str, Enum):
    billing = "billing"
    technical = "technical"
    general = "general"


class Sentiment(str, Enum):
    positive = "positive"
    negative = "negative"
    neutral = "neutral"


class CustomerMessage(BaseModel):
    name: str = Field(description="The customer's name")
    issue_type: IssueType
    sentiment: Sentiment


good = CustomerMessage(name="Raj", issue_type="billing", sentiment="negative")
print("Valid:", good.model_dump())

Valid: {'name': 'Raj', 'issue_type': <IssueType.billing: 'billing'>, 'sentiment': <Sentiment.negative: 'negative'>}


## 2. Validation catching a bad value

This is what "validate the output" (`notes.md` Section 2) actually looks like â€” Pydantic rejects
anything that doesn't match the schema, with a specific, actionable error message.

In [2]:
try:
    bad = CustomerMessage(name="Raj", issue_type="not_a_real_type", sentiment="negative")
except ValidationError as e:
    print("Validation caught the error:")
    print(e)

Validation caught the error:
1 validation error for CustomerMessage
issue_type
  Input should be 'billing', 'technical' or 'general' [type=enum, input_value='not_a_real_type', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/enum


## 3. Retry with exponential backoff

Simulating a flaky call that fails twice before succeeding â€” watch the wait time grow between
attempts, exactly as `notes.md` Section 2 describes.

In [3]:
import time


def flaky_call(attempt_log):
    """Fails the first 2 times, succeeds on the 3rd -- simulating a real timeout/rate-limit."""
    attempt_log.append(1)
    if len(attempt_log) < 3:
        raise TimeoutError("simulated timeout")
    return "success!"


def call_with_retry(fn, max_attempts=5, base_delay=0.3):
    attempt_log = []
    for attempt in range(1, max_attempts + 1):
        try:
            return fn(attempt_log)
        except TimeoutError as e:
            wait = base_delay * (2 ** (attempt - 1))
            print(f"  attempt {attempt} failed ({e}), retrying in {wait:.2f}s")
            time.sleep(wait)
    raise RuntimeError("all retries exhausted -- this is where a fallback would kick in")


result = call_with_retry(flaky_call)
print("Result:", result)

  attempt 1 failed (simulated timeout), retrying in 0.30s


  attempt 2 failed (simulated timeout), retrying in 0.60s


Result: success!


## 4. (Optional) Getting structured output from a real model

Needs an API key in `.env` (`OPENAI_API_KEY` or `ANTHROPIC_API_KEY`). Without one, this cell explains
what it would do instead of failing.

In [4]:
import os

from dotenv import load_dotenv

load_dotenv()

has_key = bool(os.getenv("OPENAI_API_KEY") or os.getenv("ANTHROPIC_API_KEY"))

if not has_key:
    print("No API key found. With one set, this cell would run:")
    print()
    print('  model = ChatOpenAI(model="gpt-4o-mini")')
    print('  structured_model = model.with_structured_output(CustomerMessage)')
    print('  structured_model.invoke("Hi, I\'m Raj and I was charged twice for my order, please help!")')
    print()
    print("...and get back a CustomerMessage object directly -- no manual JSON parsing.")
else:
    if os.getenv("OPENAI_API_KEY"):
        from langchain_openai import ChatOpenAI
        model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    else:
        from langchain_anthropic import ChatAnthropic
        model = ChatAnthropic(model="claude-3-5-haiku-latest", temperature=0)

    structured_model = model.with_structured_output(CustomerMessage)
    result = structured_model.invoke(
        "Hi, I'm Raj and I was charged twice for my order, please help!"
    )
    print(result)

No API key found. With one set, this cell would run:

  model = ChatOpenAI(model="gpt-4o-mini")
  structured_model = model.with_structured_output(CustomerMessage)
  structured_model.invoke("Hi, I'm Raj and I was charged twice for my order, please help!")

...and get back a CustomerMessage object directly -- no manual JSON parsing.


## 5. Deploying it as an API

See [`api_example.py`](api_example.py) â€” a small FastAPI app with a schema-validated `/ask` endpoint,
built and tested in this same session (including the "no API key configured" error path) using
FastAPI's `TestClient`, with no server actually running. To try it live:

```bash
uvicorn api_example:app --reload
curl -X POST http://127.0.0.1:8000/ask -H "Content-Type: application/json" -d "{\"question\": \"How do I get a refund?\"}"
```

## Try it yourself

- Add a 4th `IssueType` and see what happens when you pass an old value that's no longer valid.
- Change `call_with_retry`'s `max_attempts` to less than the number of failures `flaky_call` needs,
  and watch it raise instead of eventually succeeding.
- Run `api_example.py` with `uvicorn` and hit `/health` and `/ask` from a terminal or browser.